# Aplicação Estatística a problemas reais da Biologia

Nesta aula vamos analisar o crescimento de **6 microrganismos** em cultura pura, extrair parâmetros biológicos das curvas de crescimento, e aplicar testes estatísticos para comparar os organismos entre si.

**Organismos em estudo:**

| Organismo | Tipo | Patogenicidade | Ambiente natural |
|---|---|---|---|
| *E. coli* K12 | Bactéria Gram− | Comensal | Intestino humano |
| *Salmonella enterica* | Bactéria Gram− | Patogénico | Intestino, alimentos |
| *L. acidophilus* | Bactéria Gram+ | Comensal/probiótico | Intestino, vagina |
| *C. difficile* | Bactéria Gram+ | Patogénico | Intestino (oportunista) |
| *S. cerevisiae* | Levedura | Comensal/probiótico | Intestino humano, alimentos, solo |
| *C. albicans* | Levedura | Patogénico (oportunista) | Intestino, vagina |

## Curvas de Crescimento

O crescimento microbiano segue um padrão **sigmoidal** bem caracterizado, com fases distintas: **lag**, **exponencial**, **estacionária** e **declínio**. A análise quantitativa destas curvas permite extrair parâmetros biológicos que descrevem o comportamento de cada cultura.

---

### Parâmetros-chave

| Parâmetro | Símbolo | Significado biológico | Unidade |
|---|---|---|---|
| Plateau | **A** | Densidade ótica máxima atingida (capacidade de carga) | OD600 |
| Taxa de crescimento máxima | **μ_max** | Inclinação máxima da curva — quão rápido as células se dividem | OD/h |
| Fase lag | **λ** | Tempo que a cultura demora a iniciar o crescimento exponencial | horas |

---

### Porquê estes parâmetros?

Estes três parâmetros permitem responder a questões práticas importantes, como:
- **Segurança alimentar:** Quanto tempo demora um patogénico a começar a multiplicar-se num alimento? (λ)
- **Produção industrial:** Qual o organismo que atinge maior biomassa? (A)
- **Competição entre espécies:** Quem cresce mais rápido e pode dominar uma co-cultura? (μ_max)


In [ ]:
# instalar Pacotes se necessário
!pip install numpy pandas matplotlib scipy statsmodels

In [ ]:
# Importar Pacotes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# FASE 1 — Carregar e visualizar os dados

Antes de qualquer cálculo, é fundamental **olhar para os dados**. Vamos carregar o ficheiro CSV com as leituras de OD600 ao longo do tempo para as 6 estirpes (3 réplicas cada).

### 1.1 Abrir as medições de crescimento microbiano

O ficheiro `crescimento_microbiano.csv` contém leituras de **densidade ótica a 600 nm (OD600)** ao longo de 24 horas para as 6 estirpes, com 3 réplicas biológicas cada.

A OD600 é uma medida indireta da **concentração de células** em suspensão — quanto mais células, mais luz é dispersa e maior o valor de OD.

**Estrutura dos dados:**
- Cada **linha** = uma réplica de uma estirpe
- Cada **coluna** numérica = um ponto temporal (em horas)
- Colunas `Strain` e `Replicate` = identificação da amostra


In [ ]:
# Carregar o CSV numa dataframe pandas
df = XXX


In [ ]:
print("Boas práticas: ")

print("Verifica as primeiras linhas do DataFrame:")
print(XXX)


#### **Questão**

- Quantas medições temos no total? (linhas × colunas de tempo)
- Os valores de OD no tempo 0 são todos iguais? O que poderia explicar pequenas diferenças?

In [ ]:
print("Verificar quais as condições a analisar")
print("Estirpe:")
print(df[XXX].unique())

#print("Réplicas por estirpe:")
#print(XXX)


#### **Questão**
- No total quantas condições vamos analisar?

In [ ]:
print("\nEstatísticas descritivas:")
print(XXX) #describe

#### **Questão**
- Como varia o desvio padrão com o tempo?

### Atribuição de estirpes por grupo

Cada grupo altera a variável `estirpe` na célula seguinte para o seu organismo:

| Grupo | Organismo | Tipo |
|---|---|---|
| 1 | `E_coli_K12` | Bactéria comensal |
| 2 | `Salmonella_enterica` | Bactéria patogénica |
| 3 | `L_acidophilus` | Bactéria probiótica |
| 4 | `C_difficile` | Bactéria patogénica |
| 5 | `S_cerevisiae` | Levedura comensal |
| 6 | `C_albicans` | Levedura patogénica |

**Pergunta:** Antes de correr o código, o que esperam? Um patogénico cresce mais rápido que um comensal? Uma levedura cresce mais devagar que uma bactéria?


### 1.2 Filtrar os dados da vossa estirpe

Filtramos a dataframe de maneira a selecionar apenas as linhas correspondentes à estirpe escolhida, e removemos as colunas não numéricas (Strain, Replicate).

In [ ]:
estirpe = "E_coli_K12" # defina a estirpe a analisar


df_estirpe = df[df["Strain"] == estirpe] # filtrar o dataframe para a estirpe escolhida
print(df_estirpe)

# ----Filtrar a dataframe para as colunas com dados apenas
#print('Colunas de tempo')

#columns = XXX # LISTA o nome das colunas do dataframe
#time_cols = XXX  # mantém apenas as colunas de tempo (assumindo que as 2 primeiras são 'Strain' e 'Replicate')
#print(time_cols)

                        
#print(f"DataFrame filtrado para a estirpe {estirpe}:\n{df_estirpe_time.head(5)}")

### 1.3 Calcular média e desvio-padrão de todos os tempos

Para cada ponto temporal, calculamos a **média** e o **desvio-padrão** das 3 réplicas.

- A **média** dá-nos o valor central da medição
- O **desvio-padrão** indica a variabilidade entre réplicas — se é pequeno, as réplicas são consistentes


In [ ]:
# --- Usa a dataframe 'df_estirpe_time' para calcular os valores da média

print(df_estirpe)

od_mean = df_estirpe[XXX].mean().values # selecionar valores e calcular média para cada tempo
print(f"Médias calculadas por cada periodo de tempo analisado:\n{od_mean}")

#od_std  = XXX # calcular desvio-padrão para cada tempo
#print(f"Desvios-padrão:\n{od_std}")

##time_array = np.array(XXX, dtype=float) # criar array de tempos a das colunas de tempo
print(f"Array de tempos (horas): {time_array}")

#### **Questão:**

Olhando para a média dos valores, consegues antever:
- quando acaba a fase de adaptação do crescimento (lag)?
- quando começa  a fase de crescimento exponencial?
- quando se inicia a fase estacionária?

### 1.4 Representação gráfica

Vamos representar a curva de crescimento com a média ± desvio-padrão (barras de erro), mas também todas as réplicas individualmente.


In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'

fig, ax = plt.subplots(figsize=(8, 5))



# ────── Curva dos valores médios de crescimento
ax.plot(XXX, XXX, #X = time, Y= média de OD em cada tempo
        "o-", color="#3266ad", linewidth=2, markersize=5, label='Média, n=3')

# ────── Curva dos valores de cada réplicas individual
print(df_estirpe)
#for i, row in XXX:
    #print(row)
    #ax.plot(###, row[XXX], #X = time, Y= valores de OD em CADA TEMPO
    #       "o-", linewidth=1, markersize=3, alpha=0.6, label=row["Replicate"]) # label usando o nome da réplica


# ────── Banda de desvio padrão
#ax.fill_between(XXX, XXX, XXX,  # X = time, Y_baixo = valor mais baixo (média - std), Y_alto = valor mais alto (média + std)
#                color="#3266ad", alpha=0.2, label="± SD")

#ax.set_xlabel("Tempo (horas)")
#ax.set_ylabel("OD600")
#ax.set_title("Estimativa gráfica dos parâmetros")
#ax.legend()

#mostra a imagem e fecha, permitindo a continua alteração da imagem
display(fig)
plt.close()

### **Questão**
- Conseguem identificar as fases lag, exponencial e estacionária?
- Como se comparam as diferentes réplicas individualmente com a média e desvio padrão?

# FASE 2 — Calcular os parâmetros de crescimento

Agora vamos passar da observação ao cálculo. Para cada estirpe, vamos estimar os 3 parâmetros do modelo de Gompertz diretamente a partir dos dados:

| Passo | O que vamos fazer | Parâmetro | 
|-------|-------------------|-----------|
| 2.1 | Estimar o plateau (média dos últimos pontos) | **A** |
| 2.2 | Calcular a taxa máxima de crescimento (ΔOD/Δt) | **μ_max** |
| 2.3 | Estimar a fase lag a partir do ponto de inflexão | **λ** |

Cada grupo vai analisar **um microrganismo** e no final juntamos os resultados de todos.

### 2.1 Crescimento máximo — Plateau ou Fase Estacionária (A)

O plateau (A) representa a **densidade ótica máxima** que a cultura atinge — o ponto em que o crescimento estabiliza porque os nutrientes se esgotam ou os produtos tóxicos se acumulam.

Estimamos A como a **média dos últimos pontos temporais**, onde a curva já estabilizou.

**O que determina o valor de A?**
- **Capacidade de carga do meio:** quantidade de nutrientes disponíveis
- **Tolerância a produtos metabólicos:** ácidos, etanol, toxinas que se acumulam
- **Tamanho celular:** a mesma OD600 pode corresponder a números diferentes de células conforme o organismo (bactérias são mais pequenas que leveduras)

**Questão:** 
- Se duas estirpes crescem no mesmo meio, mas uma atinge um plateau mais alto, o que pode explicar a diferença?

In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'


# ─────────────── Plateau (A) ─────────────────────────────────────────
# ────── Calcular o plateu médio
valor_incial_A = XXX # em que valor a reta entra em A?
valores_A = od_mean[valor_incial_A:] # selecionar os valores a partir do tempo onde a curva parece estabilizar (usar o array de médias)
print(f"Valores selecionados: {valores_A}")

#A_estimado = XXX # calcular a média dos valores para obter a estimativa do plateau
#print(f"\nEstimativas gráficas para {estirpe}:")
#print(f"  Plateau (A) média ≈ {A_estimado}")

#visualiza o valor médio do plateau no gráfico
#ax.axhline(A_estimado, color="gray", linestyle="--", label=f"Plateau ≈ {A_estimado}") 
#ax.legend()

# ────── Calcular o plateau para cada réplica individualmente
for i, row in df_estirpe.iterrows():      # loop por cada réplica usando df_estirpe
    od_rep = row[time_cols].XXX.XXX(XXX)  # seleciona as colunas de valores → guarda os values de OD →  type float
    od_A_rep = XXX                        # últimos 3 valores (horas 16, 20, 24)
    A_rep = XXX                           # média → estimativa do plateau
    print(f"  {row['Replicate']}: Plateau ≈ {A_rep:.3f}")


display(fig)
plt.close()

### 2.2 Taxa de crescimento máxima (μ_max)

A taxa de crescimento máxima corresponde à **inclinação máxima** da curva — o momento em que as células se estão a dividir mais rapidamente (fase exponencial).

Calculamos μ_max como a derivada numérica da curva:

$$\mu_{max} = \max\left(\frac{\Delta OD}{\Delta t}\right)$$

Ou seja, para cada par de pontos consecutivos calculamos a variação de OD por hora (ΔOD/Δt) e ficamos com o **valor máximo**.

**O que influencia μ_max?**
- **Tipo de divisão celular:** bactérias (fissão binária, ~20 min) vs leveduras (gemulação, ~90 min)
- **Meio de cultura:** disponibilidade de nutrientes, temperatura, pH
- **Metabolismo:** aeróbio vs anaeróbio, fermentativo vs respiratório


In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'


# ─────────────── Taxa máxima (mu) ────────────────────────────────────

# ────── Calcular o mu médio
# -- np (numpy é usado para trabalhar com arrays e fazer operações amtemáticas)
dod = np.diff(od_mean) # calcular a diferença entre os valores de OD para obter a taxa de crescimento
print(f"Taxa de crescimento (ΔOD): {dod}")

dt = np.diff(time_array) # calcular intervalo de tempo
print(f"Intervalos de tempo (Δt): {dt}")

#dod_dt = XXX # aplicar a fórmula da taxa de crescimento (ΔOD/Δt)
#print(f"Taxa de crescimento por hora (ΔOD/Δt): {dod_dt}")

#mu_estimado  = XXX # encontrar o valor máximo de μ_max
#idx_max = np.argmax(dod_dt) # encontrar o índice no array no valor máximo
#time_mu_max  = time_array[idx_max] # para identificar o tempo correspondente
#print(f"O crescimento máximo de {mu_estimado} OD/h ocorre no tempo de {time_mu_max} horas")

# ────── Calcular mu para cada réplica individualmente
#for i, row in df_estirpe.iterrows():
#    od_rep      = XXX # extrair os valores de OD da réplica
#    dod_dt_rep  = XXX # calcular a taxa de crescimento da réplica
#    mu_rep      = XXX # estimar μ_max da réplica
#    print(f"  {df_estirpe.loc[i, 'Replicate']}: μ_max ≈ {mu_rep} OD/h")


# --- plot no gráfico
#ax.plot(time_array[idx_max] +1, dod_dt[idx_max], "o", color="red", 
#        label=f"μ_max ≈ {mu_estimado} OD/h")
#ax.legend()

#print(f"Estimativas gráficas para {estirpe}:")
#print(f"  Plateau (A)  ≈ {A_estimado:.3f}")
#print(f"  μ_max        ≈ {mu_estimado:.3f} OD/h")

display(fig)
plt.close()

### 2.3 Estimativa da Fase Lag (λ)

A fase lag é calculada a partir do ponto de inflexão da curva:

$$\lambda = t_{inf} - \frac{OD_{inf} - OD_0}{\mu_{max}}$$

Onde:
- **lambda** — tempo da fase lag (se λ = 2 horas, significa que a cultura ficou cerca de 2 horas em adaptação antes de crescer rapidamente)
- **t_inf** — tempo no ponto de inflexão (onde o crescimento é mais rápido)
- **OD_inf** — valor de OD nesse ponto
- **OD_0** — valor de OD no tempo zero
- **μ_max** — taxa de crescimento máxima (calculada no passo anterior)


#### **Questão:** 
- Um organismo com uma fase lag longa é vantajoso ou desvantajoso? Depende do contexto?
-- Num probiótico, uma lag curta pode ser benéfica (coloniza rapidamente)
-- Num patogénico alimentar, a lag dá-nos uma "janela de segurança" antes da contaminação

In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'



# ────── Calcular lam da média
threshold = od_mean[0] + 0.1 # OD inicial + margem para ignorar ruído
print(f"Limite de crescimento em fase Lag: {threshold}")

#array_valores_acima_threshold = XXX > XXX # cria um array booleano (True/False) dos valores de od_mean acima do threshold defnido
#print(f"Valores acima do threshold: {array_valores_acima_threshold}")

#valores_tempo_acima_threshold = XXX[XXX] # Retira os valores de tempo onde True
#print(f"Tempos acima do crescimento lag: {valores_tempo_acima_threshold}")

#lam_estimado = XXX # primeiro tempo onde OD ultrapassa o threshold
#print(f"Tempo que o crescimento sai de fase lag: {lam_estimado}")

# ────── Calcular lam para cada réplica individualmente
# for i, row in df_estirpe_time.iterrows():
   # od_rep  = row[time_cols].values # extrair os valores de OD da réplica
   # lam_rep = time_array[od_rep > od_rep[0] + 0.1][0] # primeiro tempo onde OD ultrapassa o threshold
   # print(f"  {df_estirpe.loc[i, 'Replicate']}: Lag ≈ {lam_rep} h")

#ax.axvline(lam_estimado, color="orange", linestyle="--", label=f"Lag ≈ {lam_estimado}h") # representar graficamente
#ax.legend()

#print(f"Estimativas gráficas para {estirpe}:")
#print(f"  Plateau (A)  ≈ {A_estimado}")
#print(f"  μ_max        ≈ {mu_estimado} OD/h")
#print(f"  Fase lag     ≈ {lam_estimado} h")

display(fig)
plt.close()

# FASE 3 — Comparar o crescimento das diferentes estirpes

### 3.1 Tabela resumo dos parâmetros de crescimento

Depois de estimados os parâmetros para todas as estirpes, compilamos os resultados numa tabela com a **média ± desvio-padrão** (3 réplicas) de cada parâmetro:

| Parâmetro | O que representa | Como foi estimado |
|---|---|---|
| **A** (mean ± sd) | Plateau — densidade máxima atingida | Média dos últimos 5 pontos temporais |
| **μ_max** (mean ± sd) | Taxa de crescimento máxima | Máximo de ΔOD/Δt na fase exponencial |
| **λ** (mean ± sd) | Fase lag — tempo até iniciar crescimento | Fórmula: λ = t_inf − (OD_inf − OD₀) / μ_max |

A tabela resultante terá este formato — **uma linha por estirpe, com média e desvio-padrão de cada parâmetro**:

| Strain | Replicate | A_mean | A_sd | mu_mean | mu_sd | lam_mean | lam_sd |
|---|---|---|---|---|---|---|---|
| C_albicans | ... | ... | ... | ... | ... | ... | ... |
| C_difficile | ... | ... | ... | ... | ... | ... | ... |
| E_coli_K12 | ... | ... | ... | ... | ... | ... | ... |
| L_acidophilus | ... | ... | ... | ... | ... | ... | ... |
| S_cerevisiae | ... | ... | ... | ... | ... | ... | ... |
| Salmonella_enterica | ... | ... | ... | ... | ... | ... | ... |



In [ ]:
# --- Para o cálculo de todos os dados usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'

#────── Criar dataframe vazia para guardar os resultados
df_resultados = pd.DataFrame(columns = ["Strain", "Replicate", "A", "mu", "lam"])

#────── Criar função para calcular os diferentes parâmetros de crescimento
def calcular_parametros(t, od): # cria função para calcular os parÂmetros de crescimento
    
    # Plateau — média dos últimos 20% dos pontos
    n_plateau = int(0.2 * len(od))
    A = od[-n_plateau:].mean()

    # mu máximo

    # Fase lag — threshold

    return A, mu, lam

#────── Calcular e armazenar os parâmetros de crescimento para todas as estirpes
for estirpe in XXX: # iterar por estirpe
    df_estirpe = XXX # filtrar por estirpe
    df_estirpe_time = XXX # manter apenas colunas com valores de crescimento

    for i, row in df_estirpe.iterrows(): # iterar df_estirpe 

        od_rep = XXX # valores de OD da linha 
        A, mu, lam = calcular_parametros(time_array, od_rep)

        #guardar resultados
        df_resultados.loc[i, "Strain"]    = estirpe
        df_resultados.loc[i, "Replicate"] = df_estirpe.loc[i, "Replicate"]
        df_resultados.loc[i, "A"] = A
        df_resultados.loc[i, "mu"] = mu
        df_resultados.loc[i, "lam"] = lam


print("Parâmetros por réplica:")
print(df_resultados)

#────── Colapsar em valores médios, calculando o desvio padrão
#df_resumo = df_resultados.groupby("Strain").agg(  #agrupar as diferentes linhas pela coluna strain, agregando...
#    A_mean=("A", "mean"), A_sd=("A", "std"), # coluna A_mean em que agregamos os valores de A pela média e coluna A_std em que agregamos os valores de A pelo desvio padrão
#    XXX # repete para mu
#    XXX # repete para lam
#).round(3)
#print("\nResumo (média ± SD por estirpe):")
#print(df_resumo)

#### **Questão:** 
- Qual a estirpe com maior variabilidade (SD alto) em cada parâmetro? O que pode explicar isso?

### 3.1 Simulação das curvas de crescimento — Modelo de Gompertz

Tendo os 3 parâmetros estimados (A, μ_max, λ) para cada estirpe, podemos reconstruir a curva de crescimento completa usando o **modelo de Gompertz modificado**:

$$OD(t) = A \cdot \exp\left(-\exp\left(\frac{\mu_{max} \cdot e}{A} \cdot (\lambda - t) + 1\right)\right)$$

Isto permite-nos **sobrepor todas as estirpes no mesmo gráfico** e comparar visualmente.

In [ ]:
# Para a simulação usa a dataframe df_resumo
print(df_resumo)

# ────── Cria o gráfico
fig_all, ax_all = plt.subplots(figsize=(10, 6))

# ────── Função que faz a simulação da curva por estirpe
def gompertz(t, A, mu, lam):
    return XXX #Tenta completar a fórmula. Lembra-te que o np permite operações matemáticas (np.exp)

# ---np.linspace(start, stop, num) — cria um array de num valores igualmente espaçados entre start e stop.
t_curva = np.linspace(0, time_array[-1], 200)
print(f"Linha temporal criada: {t_curva}")

# ──────simula a curva por cada estirpe
#for estirpe in df_resumo.XXX.unique(): 
#    params   = df_resumo.loc[estirpe]
#    print(params)                                      #retira os valores da estirpe da df_resumo
#    od_curva = gompertz(XXX, XXX, XXX, XXX)                                #insere os valores necessários para a função
#    ax_all.plot(XXX, XXX, linewidth=2, label=estirpe.replace("_", " "))    #plota a linha no gráfico - X = tempo, Y = ods calculados por gompertz

#ax_all.set_xlabel("Tempo (horas)")
#ax_all.set_ylabel("OD600")
#ax_all.legend()
#ax_all.grid(alpha=0.3)
#plt.tight_layout()
#plt.show()

#### **Questão:** 

- Quem arranca primeiro? (λ mais curto = curva desloca-se para a esquerda)
- Quem cresce mais rápido? (μ_max maior = curva mais íngreme)
- Quem atinge maior densidade? (A maior = plateau mais alto)

- Olhando para o gráfico, consigam identificar os dois grandes grupos (bactérias vs leveduras). O que os distingue?

### 3.2 Visualização comparativa de parâmetros de crescimento

Para facilitar a comparação dos parâmetros de crescimento, é possivel criar visualização de gráfico de barras de todas as estirpes.

In [ ]:
#  Para a visualização usa a dataframe 'df_resumo'
print(df_resumo)

# ──────Cria figura com espaço para 3 gráficos
fig_bar, ax_bar = plt.subplots(1, 3, figsize=(14, 5))

# --- No params_plot definimos que o que queremos ver em cada gráfico
# --- mean_col, sd_col, titulo, unidade 
params_plot = [
    ("A_mean",   "A_sd",   "Plateau (A)",    "OD600"),
    ("mu_mean",  "mu_sd",  "Taxa máxima (µ)", "OD/h"),
    ("lam_mean", "lam_sd", "Fase lag (λ)",    "horas"),
]

estirpes = XXX # retira as estirpes da df_resumo - index
print(f"lista de estirpes: {estirpes}")

# ────── Criar um gráfico de barras para cada parâmetro e cada estirpe
# --- O zip vai retirar os todos os valores em ax_bar e params_plot
#for ax, (mean_col, sd_col, titulo, unidade) in zip(ax_bar, params_plot):
    #valores = XXX #Retira da df_resumo os values da média - media_col
    #erros   = XXX #Retira da df_resumo os values da média - sd_co

    #bars = ax.bar(XXX, XXX, yerr=XXX) # X = estirpes, Y = valores de média, yerr = valores para as barras de erro

    #ax.set_xticks(range(len(estirpes)))
    #ax.set_xticklabels(estirpes, fontsize=8, rotation=45)
    #ax.set_title(titulo)
    #ax.set_ylabel(unidade)


#plt.suptitle("Comparação de parâmetros de crescimento (média ± SD)", fontsize=13)
#plt.tight_layout()
#plt.show()

#### **Questão:** 
- Comparem A com μ_max: os organismos que crescem mais rápido são os que atingem maior densidade?

### 3.3 Os parâmetros de crescimento são estatisticamente diferentes entre estirpes?

Visualmente, as curvas e gráficos de barras parecem diferentes — mas será que essas diferenças são **estatisticamente significativas** ou podem ser apenas variabilidade experimental?

Para responder, vamos aplicar:

1. **ANOVA One-way** — testa se existe pelo menos uma estirpe diferente das outras (para cada parâmetro)
2. **Tukey HSD** (post-hoc) — se a ANOVA for significativa, identifica **quais pares** de estirpes são diferentes

Relembrando:
- **p < 0.05** → diferença significativa (rejeitamos H₀)
- **p ≥ 0.05** → sem evidência de diferença (não rejeitamos H₀)


In [3]:
# Valores a usar estão na dataframe 'df_resultados' e a lista de estirpes na variável 'estirpe'

print(df_resultados)
print(estirpe)

for param, titulo in [("A", "Plateau (A)"), ("mu", "Taxa máxima (µ)"), ("lam", "Fase lag (λ)")]:

    # ── Preparar Valores ────────────────────
    # criar um dicionário com os valores do parâmetro por estirpe
    grupos = {}

    for e in estirpes:
        df_estirpe_params = df_resultados[XXX[XXX] == XXX]  # filtrar para a estirpe
        grupos[e] = XXX[XXX].XXX.astype(float) # array com os 3 parametros (.values) das réplicas e converte em float (.astype)
    print(f"Grupos: {grupos}")

    # ── ANOVA entre todas as estirpes ────────────────────
    #f_stat, p_anova = stats.f_oneway(*grupos.values()) # * grupos = a stats.f_oneway(grupos["E_coli_K12"], grupos["Salmonella_enterica"], grupos["L_acidophilus"], ...)
    # --- *grupos.values(): [1.5065  1.51975 1.58   ] [2.39325 2.00575 2.15375] [2.87975 2.7845  2.7195 ] [1.89175 2.16425 1.93875] [2.21225 2.167   2.259  ] [2.55375 2.54975 2.77125]
    #print(f"\nANOVA (todas as estirpes): F={f_stat}, p={p_anova}")

    # ── Tukey HSD post-hoc ───────────────────────────────
   # valores = np.concatenate(list(grupos.values()))
    #  --- np.concatenate(list(grupos.values())) - [1.5065  1.51975 1.58    2.39325 2.00575 2.15375 2.87975 2.7845  ....
    #labels = np.repeat(list(grupos.keys()), 3) # diz que cada 3 valores são a réplica de uma espécie
    #tukey   = pairwise_tukeyhsd(valores, labels, alpha=0.05)
    #print("\nTukey HSD:")
    #print(tukey.summary())

NameError: name 'df_resultados' is not defined

#### **Questões**

- Para qual dos 3 parâmetros (A, μ, λ) existem mais diferenças significativas entre estirpes?
- Há estirpes que não são significativamente diferentes entre si? Quais? Faz sentido biologicamente?
- *E. coli* e *Salmonella* (ambas Enterobacteriaceae) têm parâmetros semelhantes? E *S. cerevisiae* e *C. albicans* (ambas leveduras)?

### 3.4 Os parâmetros de crescimento são estatisticamente diferentes entre Bactérias e Leveduras

Agora agrupamos os organismos por **tipo** (bactérias vs leveduras) e testamos se há diferenças entre os dois grupos.


In [ ]:
# ══════════════════════════════════════════════════════════
# ANÁLISE POR GRUPO — bactérias vs leveduras
# ══════════════════════════════════════════════════════════
bacterias = ["E_coli_K12", "Salmonella_enterica", "L_acidophilus", "C_difficile"]
leveduras = ["S_cerevisiae", "C_albicans"]

for param, titulo in [("A", "Plateau (A)"), ("mu", "Taxa máxima (µ)"), ("lam", "Fase lag (λ)")]:

    # criar um dicionário com os valores do parâmetro por estirpe
    grupos = {}
    for e in estirpes:
        df_estirpe_params = df_resultados[df_resultados["Strain"] == e]  # filtrar para a estirpe
        grupos[e] = df_estirpe_params[param].values.astype(float)  # array com os 3 valores das réplicas

    #DIFERENÇA
    # juntar todos os valores de bactérias num único array
    vals_bact = np.concatenate([grupos[e] for e in bacterias])
    # juntar todos os valores de leveduras num único array
    vals_lev  = XXX

    # ── t-test bactérias vs leveduras ────────────────────
    # -- ttest_ind(valores de crescimento das bacterias, valores de crescimento das leveduras)
    t_stat, p_ttest = stats.ttest_ind(XXX, XXX)
    print(f"\nt-test bactérias vs leveduras para {titulo}: t={t_stat:.3f}, p={p_ttest:.4f}")
    print(f"  bactérias: média={vals_bact.mean():.3f} ± {vals_bact.std():.3f}")
    print(f"  leveduras: média={vals_lev.mean():.3f} ± {vals_lev.std():.3f}")

#### **Questões**

- As bactérias atingem um plateau mais alto que as leveduras? É significativo?
- As leveduras demoram mais a iniciar o crescimento? O que pode explicar isso biologicamente?
- A taxa de crescimento é diferente? As bactérias dividem-se por **fissão binária** (rápido) e leveduras por **gemulação** (mais lento)


### 3.4 Os parâmetros de crescimento são estatisticamente diferentes entre Patogénicas vs Comensais

Será que os organismos **patogénicos** crescem de forma diferente dos **comensais**? Separamos a análise para bactérias e leveduras.

| Bactérias patogénicas | Bactérias comensais | Leveduras patogénicas | Leveduras comensais |
|---|---|---|---|
| *Salmonella*, *C. difficile* | *E. coli* K12, *L. acidophilus* | *C. albicans* | *S. cerevisiae* |


In [ ]:
# ── Bactérias patogénicas vs comensais ──────────────────
bact_patogenicas = ["Salmonella_enterica", "C_difficile"]
bact_comensais   = ["E_coli_K12", "L_acidophilus"]
# ── Leveduras patogénicas vs comensais ──────────────────
lev_patogenicas  = ["C_albicans"]
lev_comensais    = ["S_cerevisiae"]

for param, titulo in [("A", "Plateau (A)"), ("mu", "Taxa máxima (µ)"), ("lam", "Fase lag (λ)")]:
    
    #separador visual
    print(f"\n{'═'*55}")
    print(f"{titulo}")
    print(f"{'═'*55}")

    # criar um dicionário com os valores do parâmetro por estirpe
    XXX
    for e in XXX: #loop pelas estirpes
        df_estirpe_params =XXX  # filtrar para a estirpe
        grupos[e] = XXX        # array com os 3 valores das réplicas

    # ── Bactérias patogénicas vs comensais ───────────────
    vals_bact_pat = XXX     # juntar valores das patogénicas
    vals_bact_com = XXX     # juntar valores das comensais
    t_stat, p_ttest = XXX   # t-test
    print(f"\nt-test bactérias patogénicas vs comensais para {titulo}: t={t_stat:.3f}, p={p_ttest:.3f}")
    print(f"  patogénicas: média={vals_bact_pat.mean():.3f} ± {vals_bact_pat.std():.3f}")
    print(f"  comensais:   média={vals_bact_com.mean():.3f} ± {vals_bact_com.std():.3f}")

    # ── Leveduras patogénicas vs comensais ───────────────
    vals_lev_pat =XXX       # juntar valores das patogénicas
    vals_lev_com = XXX      # juntar valores das comensais
    t_stat, p_ttest =XXX    # t-test
    print(f"\nt-test leveduras patogénicas vs comensais para {titulo}: t={t_stat:.3f}, p={p_ttest:.3f}")
    print(f"  patogénicas: média={vals_lev_pat.mean():.3f} ± {vals_lev_pat.std():.3f}")
    print(f"  comensais:   média={vals_lev_com.mean():.3f} ± {vals_lev_com.std():.3f}")

#### **Questões**

- Há diferenças significativas entre patogénicos e comensais dentro das bactérias? E dentro das leveduras?
- Se *C. albicans* (patogénica) cresce mais devagar que *S. cerevisiae* (comensal), que importância pode ter?

# FASE 4 — Conclusões

Respondam em grupo às seguintes questões:

1. **Ranking de crescimento:** Ordenem os 6 organismos do mais rápido ao mais lento. Coincidem com o que esperavam?

2. **Bactérias vs leveduras:** Quais as principais diferenças? São estatisticamente significativas para todos os parâmetros?

3. **Patogénicos vs comensais:** A patogenicidade está associada a um padrão de crescimento diferente?

4. **Previsão para co-cultura:** Com base nestes resultados, o que esperam que aconteça quando os 6 organismos crescerem juntos?
   - Quem vai dominar?
   - Quem pode ser eliminado?
   - Que mecanismos de interação podem ocorrer? (competição, produção de bacteriocinas, alteração de pH...)

---

**Na próxima aula:** Vamos usar **RT-PCR** com primers específicos para verificar a presença de cada espécie numa co-cultura real — e ver se as vossas previsões estavam corretas!
